# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
Data/
├── EEG/ID_x
│   ├── ... fichiers level_1, level_2, ..., level_9
│   ├── ... fichiers baseline
│   └── ... fichiers filtered_*
│
├── EEG_Features_10s/
│   ├── ID1_EEG_features.csv
│   ├── ID2_EEG_features.csv
│   └── ...
│
├── Labels/
│   ├── ID1.csv
│   ├── ID2.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
Data/
├── Normalized_Features_10s/
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
│
├── Normalized_Features_10s_With_Label/(avec colonnes Level et Label)
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
```

## Question 

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features_10s`  ?

### Réponse 

Trois raisons bloquantes :

1. **Variabilité inter-sujet.** Les puissances EEG absolues dépendent du sujet (anatomie, impédance des électrodes Muse S). Sans normalisation par la baseline propre à chaque participant, le modèle apprend l'identité du sujet, pas la charge cognitive — c'est précisément la raison pour laquelle CL-Drive (Angkan et al., 2023) fournit des enregistrements *baseline* en plus des *task* (§3 du papier).
2. **Mélange baseline / tâche.** Les CSV de `EEG_Features_10s/` contiennent aussi les fenêtres de baseline, qui ne sont pas des exemples à classer mais la référence physiologique du sujet.
3. **Absence de cible.** Aucune colonne `Level` ni `Label` (PAAS) dans ces fichiers ⇒ apprentissage supervisé impossible avant les sections 2 et 4.

À cela s'ajoute la nécessité d'une standardisation z-score (échelles hétérogènes entre PSD Welch et entropie spectrale, cf. TD4), traitée après le split train/test pour éviter le data leakage (§8).

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from pathlib import Path
from collections import defaultdict

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, LeaveOneGroupOut
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')

#  Chemins
BASE_PATH           = Path('../data')

EEG_FEATURE_DIR     = BASE_PATH / 'EEG_Features_10s'
LABEL_DIR           = BASE_PATH / 'Labels'
NORMALIZED_ROOT     = BASE_PATH / 'Normalized_Features_10s'
NORMALIZED_EEG_DIR  = NORMALIZED_ROOT / 'EEG'
LABELED_ROOT        = BASE_PATH / 'Normalized_Features_10s_With_Label'
LABELED_EEG_DIR     = LABELED_ROOT / 'EEG'

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

METADATA_COLUMNS = ['Participant', 'File', 'Window', 'Start_Time', 'End_Time', 'Channel']

print('Dossiers créés :')
print(' ', NORMALIZED_EEG_DIR)
print(' ', LABELED_EEG_DIR)

## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d’obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d’entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse 

La baseline est l'état physiologique de **repos** du sujet (yeux ouverts, pas de tâche), enregistré juste avant chaque scénario de conduite dans CL-Drive. Diviser une feature de tâche par sa baseline (ratio sans dimension) sert à :

1. **Supprimer la composante propre au sujet.** Les puissances EEG absolues (PSD Welch) dépendent fortement de l'anatomie crânienne, de l'impédance des électrodes Muse S et de l'état du jour. Cette part n'est pas liée à la charge cognitive et noierait le signal d'intérêt. Le ratio $x / \mu_{baseline}$ rend les sujets **comparables entre eux**, ce qui est indispensable en LOSO (§9).
2. **Mesurer une *variation* relative à un état de référence connu.** En électrophysiologie, l'activation cognitive se traduit par exemple par une **désynchronisation alpha** (chute de la puissance alpha vs repos) et une **augmentation theta frontale** — phénomènes définis par construction comme un écart à la baseline, pas comme une valeur absolue.
3. **Stabiliser le signal contre les dérives lentes** (impédance, conditions ambiantes, somnolence) : la baseline est ré-enregistrée à chaque niveau, donc le ratio est local dans le temps.

Sans cette étape, le modèle apprendrait surtout à identifier *qui* porte le casque, pas son niveau de charge cognitive.

In [15]:
def get_feature_columns(df, metadata_columns=METADATA_COLUMNS):
    """
    Retourne les colonnes numeriques correspondant aux features.
    Les colonnes de metadonnees ne doivent pas etre normalisees.
    """
    return [
        c for c in df.select_dtypes(include="number").columns
        if c not in metadata_columns
    ]


def compute_baseline_averages(feature_dir):
    """
    Calcule, pour chaque Participant, la moyenne de baseline de chaque feature.

    Layout reel : les lignes baseline et task sont DANS le meme CSV par sujet
    (<ID>_features.csv), distinguees par la colonne 'File' (ex: 'eeg_baseline_level_3.csv'
    vs 'eeg_data_level_3.csv'). On filtre donc sur le contenu de cette colonne.
    """
    baseline_avgs = {}
    for csv_path in sorted(Path(feature_dir).glob("*.csv")):
        df = pd.read_csv(csv_path)
        baseline_df = df[df["File"].astype(str).str.contains("baseline", case=False)]
        if baseline_df.empty:
            continue
        participant = df["Participant"].iloc[0]
        feature_cols = get_feature_columns(baseline_df)
        baseline_avgs[participant] = baseline_df[feature_cols].mean()
    return baseline_avgs


def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline pour le sujet considere.
    Une moyenne de baseline nulle est remplacee par NaN pour eviter une division par zero.
    """
    df_norm = df.copy()
    mu = baseline_avgs[participant_id]
    cols = [c for c in get_feature_columns(df_norm) if c in mu.index]
    df_norm[cols] = df_norm[cols].div(mu[cols].replace(0, np.nan))
    return df_norm


def run_eeg_normalization():
    """
    Genere les fichiers du dossier Normalized_Features_10s/EEG
    a partir du dossier EEG_FEATURE_DIR.

    Les fenetres baseline servent uniquement a calculer le profil de reference
    du sujet, elles ne sont PAS recopiees dans la sortie (voir section 3).
    """
    baseline_avgs = compute_baseline_averages(EEG_FEATURE_DIR)
    print(f"Profils baseline calcules pour {len(baseline_avgs)} sujets.")

    n_files = 0
    for csv_path in sorted(Path(EEG_FEATURE_DIR).glob("*.csv")):
        df = pd.read_csv(csv_path)
        df_task = df[~df["File"].astype(str).str.contains("baseline", case=False)].copy()
        if df_task.empty:
            continue
        participant = df_task["Participant"].iloc[0]
        if participant not in baseline_avgs:
            print(f"[SKIP] {participant} : pas de profil baseline disponible.")
            continue
        df_norm = normalize_by_baseline(df_task, participant, baseline_avgs)
        out_path = NORMALIZED_EEG_DIR / f"norm_{csv_path.name}"
        df_norm.to_csv(out_path, index=False)
        n_files += 1
        print(f"[OK]   {participant} : {len(df_norm)} lignes -> {out_path.name}")
    print(f"Termine : {n_files} fichiers ecrits dans {NORMALIZED_EEG_DIR}")

In [16]:
run_eeg_normalization()

Profils baseline calcules pour 21 sujets.
[OK]   1030 : 640 lignes -> norm_1030_features.csv
[OK]   1105 : 636 lignes -> norm_1105_features.csv
[OK]   1106 : 648 lignes -> norm_1106_features.csv
[OK]   1241 : 648 lignes -> norm_1241_features.csv
[OK]   1271 : 572 lignes -> norm_1271_features.csv
[OK]   1314 : 560 lignes -> norm_1314_features.csv
[OK]   1323 : 648 lignes -> norm_1323_features.csv
[OK]   1337 : 648 lignes -> norm_1337_features.csv
[OK]   1372 : 648 lignes -> norm_1372_features.csv
[OK]   1417 : 648 lignes -> norm_1417_features.csv
[OK]   1434 : 608 lignes -> norm_1434_features.csv
[OK]   1544 : 604 lignes -> norm_1544_features.csv
[OK]   1547 : 548 lignes -> norm_1547_features.csv
[OK]   1595 : 648 lignes -> norm_1595_features.csv
[OK]   1629 : 504 lignes -> norm_1629_features.csv
[OK]   1716 : 620 lignes -> norm_1716_features.csv
[OK]   1717 : 612 lignes -> norm_1717_features.csv
[OK]   1744 : 404 lignes -> norm_1744_features.csv
[OK]   1868 : 436 lignes -> norm_1868_fe

## 3. Vérification du dossier `Normalized_Features_10s/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse 

Pour trois raisons :

1. **La baseline a déjà rempli son rôle.** Elle a servi à calculer $\mu_{baseline}^{(s,f)}$ qui définit la référence du sujet. Une fois ce profil extrait, les fenêtres baseline n'apportent plus d'information.
2. **Pas de label PAAS exploitable.** Le score subjectif PAAS est attribué pendant la conduite (scénarios `eeg_data_level_*`), pas pendant les périodes de repos. Garder les baselines créerait des exemples sans cible cohérente.
3. **Information non discriminante.** En normalisant la baseline par elle-même on obtiendrait $x / \mu \approx 1$ pour toutes ses fenêtres, soit un bloc quasi constant qui biaiserait l'apprentissage vers la classe « charge faible » sans rapport avec la conduite.

En résumé : la baseline est un **outil de calibrage par sujet**, pas un exemple d'apprentissage.

In [17]:
# Verification du dossier Normalized_Features_10s/EEG

norm_files = sorted(NORMALIZED_EEG_DIR.glob("norm_*.csv"))
print(f"Fichiers norm_*.csv trouves : {len(norm_files)} / 21 attendus")
assert len(norm_files) == 21, "Nombre de fichiers normalises incorrect."

# Verification globale sur les 21 sujets
n_rows_total = 0
participants_seen = []
baseline_leak = 0
for f in norm_files:
    df = pd.read_csv(f)
    n_rows_total += len(df)
    participants_seen.append(df["Participant"].iloc[0])
    baseline_leak += df["File"].astype(str).str.contains("baseline").sum()

print(f"Total fenetres normalisees : {n_rows_total}")
print(f"Participants distincts     : {len(set(participants_seen))} / 21")
print(f"Lignes baseline residuelles: {baseline_leak} (attendu : 0)")
assert baseline_leak == 0, "Des lignes baseline ont fuite dans le dossier normalise."

# Inspection detaillee du premier sujet
sample = pd.read_csv(norm_files[0])
print(f"\nDetail {norm_files[0].name} : forme {sample.shape}")
print(f"  canaux            : {sorted(sample['Channel'].unique())}")
print(f"  fichiers sources  : {sorted(sample['File'].unique())}")
levels = sorted({int(re.search(r'level_(\d)', f).group(1)) for f in sample['File'].unique()})
print(f"  niveaux distincts : {levels}")

# Sanity check : par construction x_norm = x_task / mu_baseline.
# Le median sur l'ensemble des fenetres task doit etre du meme ordre de grandeur que 1
# si la tache n'est pas trop differente de la baseline. Des valeurs >> 1 sur certaines
# bandes (ex: theta, beta) traduisent l'activation cognitive attendue.
feature_cols = get_feature_columns(sample)
print(f"\n{len(feature_cols)} features normalisees disponibles.")
band_abs = [c for c in feature_cols if c.endswith('_abs')]
print("\nMedianes des features _abs (ratio task / baseline) :")
print(sample[band_abs].median().round(3))

sample.head()

Fichiers norm_*.csv trouves : 21 / 21 attendus
Total fenetres normalisees : 12556
Participants distincts     : 21 / 21
Lignes baseline residuelles: 0 (attendu : 0)

Detail norm_1030_features.csv : forme (640, 46)
  canaux            : ['AF7', 'AF8', 'TP10', 'TP9']
  fichiers sources  : ['eeg_data_level_1.csv', 'eeg_data_level_2.csv', 'eeg_data_level_3.csv', 'eeg_data_level_4.csv', 'eeg_data_level_5.csv', 'eeg_data_level_6.csv', 'eeg_data_level_7.csv', 'eeg_data_level_8.csv', 'eeg_data_level_9.csv']
  niveaux distincts : [1, 2, 3, 4, 5, 6, 7, 8, 9]

38 features normalisees disponibles.

Medianes des features _abs (ratio task / baseline) :
delta_abs    0.417
theta_abs    0.549
alpha_abs    0.757
beta_abs     0.890
gamma_abs    0.803
dtype: float64


,delta_abs,delta_mean,delta_max,delta_min,delta_median,theta_abs,theta_mean,theta_max,theta_min,theta_median,...,max,median,var,std,Participant,File,Window,Channel,Start_Time,End_Time
0,0.145051,0.145051,0.176770,0.102229,0.161999,0.144693,0.144693,0.079264,0.149635,0.204944,...,0.435953,1.009406,0.180306,0.497417,1030,eeg_data_level_1.csv,0,AF7,120.007812,130.003906
1,0.278198,0.278198,0.201000,0.382756,0.332687,0.383597,0.383597,0.402045,0.217913,0.396988,...,1.006359,0.865205,0.355542,0.698491,1030,eeg_data_level_1.csv,0,AF8,120.007812,130.003906
2,1.294594,1.294594,2.328772,1.103714,0.782431,1.511316,1.511316,1.042983,1.748243,1.938630,...,1.593062,1.297807,1.923925,1.624835,1030,eeg_data_level_1.csv,0,TP9,120.007812,130.003906
3,0.693162,0.693162,0.605816,0.924189,0.770291,1.433155,1.433155,1.173103,1.925799,1.668549,...,0.981913,0.913272,0.842424,1.075178,1030,eeg_data_level_1.csv,0,TP10,120.007812,130.003906
4,0.429728,0.429728,0.522642,0.353939,0.296386,0.749975,0.749975,0.458119,1.473108,0.889736,...,1.140812,1.033439,0.498608,0.827170,1030,eeg_data_level_1.csv,1,AF7,130.007812,140.003906


## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [18]:
def extract_level_from_filename(file_name):
    """
    Extrait le niveau de scenario a partir du nom de fichier.

    Exemples :
      eeg_data_level_3.csv          -> 3
      filtered_level_8.csv          -> 8
      norm_filtered_level_5.csv     -> 5
    Retourne None si aucun niveau n'est detecte.
    """
    match = re.search(r"level_(\d)", str(file_name))
    return int(match.group(1)) if match else None


def get_label_for_row(row, labels_df):
    """
    Retourne le score PAAS associe a une fenetre EEG.

    Convention TP :
      time = (Window + 1) * 10
      Level (1..9) -> colonne 'lvl_<Level>' dans labels_df
    """
    time_stamp = (int(row["Window"]) + 1) * 10
    level = row["Level"]
    if pd.isna(level):
        return np.nan
    label_col = f"lvl_{int(level)}"
    if label_col not in labels_df.columns:
        return np.nan
    match = labels_df.loc[labels_df["time"] == time_stamp, label_col]
    return match.iloc[0] if not match.empty else np.nan


def attach_labels_eeg():
    """
    Genere les fichiers du dossier Normalized_Features_10s_With_Label/EEG
    a partir des fichiers norm_*.csv et des labels PAAS dans LABEL_DIR.

    Chaque CSV de sortie contient les memes colonnes que l'entree,
    plus 'Level' et 'Label'. Les lignes sans label valide sont supprimees.
    """
    # Securite : creer le dossier de sortie si la cellule chemins n'a pas tourne.
    LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

    norm_files = sorted(NORMALIZED_EEG_DIR.glob("norm_*.csv"))
    print(f"Fichiers normalises a labelliser : {len(norm_files)}")

    n_files = 0
    n_rows_in = n_rows_out = 0
    for csv_path in norm_files:
        df = pd.read_csv(csv_path)
        participant = df["Participant"].iloc[0]

        labels_path = LABEL_DIR / f"{participant}.csv"
        if not labels_path.exists():
            print(f"[SKIP] {participant} : labels introuvables ({labels_path}).")
            continue
        labels_df = pd.read_csv(labels_path)

        df["Level"] = df["File"].apply(extract_level_from_filename)
        df["Label"] = df.apply(lambda r: get_label_for_row(r, labels_df), axis=1)

        n_rows_in += len(df)
        df = df.dropna(subset=["Label"]).copy()
        n_rows_out += len(df)

        out_path = LABELED_EEG_DIR / csv_path.name
        df.to_csv(out_path, index=False)
        n_files += 1
        print(f"[OK]   {participant} : {len(df)} lignes -> {out_path.name}")

    print(f"\nTermine : {n_files} fichiers ecrits dans {LABELED_EEG_DIR}")
    print(f"Lignes conservees : {n_rows_out} / {n_rows_in} "
          f"({n_rows_in - n_rows_out} sans label PAAS, supprimees)")

In [14]:
attach_labels_eeg()

Fichiers normalises a labelliser : 21
[OK]   1030 : 640 lignes -> norm_1030_features.csv
[OK]   1105 : 636 lignes -> norm_1105_features.csv
[OK]   1106 : 648 lignes -> norm_1106_features.csv
[OK]   1241 : 648 lignes -> norm_1241_features.csv
[OK]   1271 : 572 lignes -> norm_1271_features.csv
[OK]   1314 : 560 lignes -> norm_1314_features.csv
[OK]   1323 : 648 lignes -> norm_1323_features.csv
[OK]   1337 : 648 lignes -> norm_1337_features.csv
[OK]   1372 : 648 lignes -> norm_1372_features.csv
[OK]   1417 : 648 lignes -> norm_1417_features.csv
[OK]   1434 : 608 lignes -> norm_1434_features.csv
[OK]   1544 : 604 lignes -> norm_1544_features.csv
[OK]   1547 : 548 lignes -> norm_1547_features.csv
[OK]   1595 : 648 lignes -> norm_1595_features.csv
[OK]   1629 : 504 lignes -> norm_1629_features.csv
[OK]   1716 : 620 lignes -> norm_1716_features.csv
[OK]   1717 : 612 lignes -> norm_1717_features.csv
[OK]   1744 : 404 lignes -> norm_1744_features.csv
[OK]   1868 : 436 lignes -> norm_1868_featur

## 5. Vérification du dossier `Normalized_Features_10s_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

Level désigne le numéro du scénario de conduite (1 à 9), c'est une information sur la complexité de la tâche.

Label est le score PAAS déclaré par le participant, c'est la vérité terrain pour la classification.

Il faut conserver les deux car Level permet de retrouver l'origine d'une fenêtre et d'analyser les résultats par scénario, tandis que Label est la cible utilisée pour entraîner le modèle.

In [6]:
import pandas as pd

# Vérification du dossier Normalized_Features_10s_With_Label/EEG
files = sorted(LABELED_EEG_DIR.glob("*.csv"))
print(f"Nombre de fichiers : {len(files)}")

required_cols = ["Participant", "File", "Window", "Channel", "Start_Time", "End_Time", "Level", "Label"]

for f in files:
    df_check = pd.read_csv(f)
    missing = [col for col in required_cols if col not in df_check.columns]
    if missing:
        print(f"{f.name} : colonnes manquantes -> {missing}")
    else:
        print(f"{f.name} : OK ({len(df_check)} lignes, {len(df_check.columns)} colonnes)")

Nombre de fichiers : 0


## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [9]:
import pandas as pd

def load_labeled_eeg_dataset():
    """
    Concatène tous les fichiers CSV du dossier Normalized_Features_10s_With_Label/EEG.
    """
    # TODO : parcourir LABELED_EEG_DIR, lire les CSV, concaténer avec pd.concat.
    dfs = []
    for f in sorted(LABELED_EEG_DIR.glob("*.csv")):
        dfs.append(pd.read_csv(f))
    return pd.concat(dfs, ignore_index=True)


df = load_labeled_eeg_dataset()
print(df.shape)
df.head()

ValueError: No objects to concatenate

In [10]:
# Création des cibles de classification.
df["Label_Binary"] = df["Label"].apply(lambda x: 0 if x <= 4 else 1)

# Extension ternaire éventuelle.
df["Label_Ternary"] = df["Label"].apply(lambda x: 0 if x <= 3 else (1 if x <= 6 else 2))

print("Distribution binaire :")
print(df["Label_Binary"].value_counts())
print("\nDistribution ternaire :")
print(df["Label_Ternary"].value_counts())

NameError: name 'df' is not defined

## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

## Question

Pourquoi ne faut-il pas inclure `Participant`, `File`, `Window`, `Level` ou `Label` dans les features du modèle ?

### Réponse 

Il ne faut pas inclure Participants, File, Window, Level et Label dans les features du modèle car ce ne sont pas des mesures physiologiques. Elles servent à identifier l'origine ou la cible d'une fenêtre. Cela fausserait le modèle de les inclure car, par exemple, si le modèle apprend que le participant X a souvent une charge cognitive élevée, il mémorisera au lieu de généraliser. En particulier, le Label ne doit pas être inclus car c'est la variable qu'on cherche à prédire.

In [ ]:
# Préparation des données d’entraînement

METADATA_COLUMNS = ["Participant", "File", "Window", "Channel", "Start_Time", "End_Time", "Level", "Label", "Label_Binary", "Label_Ternary"]

feature_cols = [col for col in df.columns if col not in METADATA_COLUMNS]

X = df[feature_cols].values
y = df["Label_Binary"].values

print("Nombre d'exemples :", X.shape[0])
print("Nombre de features :", X.shape[1])
print("Exemples de features :", feature_cols[:5])

## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier

models = {
  "LDA": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearDiscriminantAnalysis()),
  ]),
  "SVM": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC())
  ]),
  "Random Forest": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier())
  ]),
  "KNN": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier())
  ]),
  "Naive Bayes": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GaussianNB())
  ]),
  "Decision Tree": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier())
  ]),
  "AdaBoost": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", AdaBoostClassifier())
  ]),
  "MLP": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(max_iter=500))
  ]),
}

## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d’évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L’opération est ensuite répétée sur l’ensemble des sujets disponibles afin d’obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse 

Le LOSO est plus difficile que le 10-fold car en 10-fold, les données d’un même sujet peuvent se retrouver à la fois dans le train et dans le test. Le modèle voit donc déjà des patterns très similaires et la tâche devient plus facile. En LOSO, le sujet de test est complètement absent de l’entraînement, donc le modèle doit généraliser à une nouvelle personne.

In [ ]:
# TODO : 10 fold cross-validation, and Leave One Subject Out.

loso = LeaveOneSubjectOut()
results_loso = {}

results_10fold = {}
k=10
kfolds = StratifiedKFold(n_splits=k, random_state=42, shuffle=True)
groups = df['subject_id']
# 1. En-tête pour rendre l'affichage lisible
print(f"{'Modèle':<18} | {'10-Fold Acc':>11} {'10-F Macro':>11} {'10-F Weight':>11} | {'LOSO Acc':>10} {'LOSO Macro':>11} {'LOSO Weight':>11}")
print("-" * 96)

for name, model in MODELS.items():
    # Pipeline pour éviter le data leakage avec le StandardScaler
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])

    # Validation 10-fold
    scores10fold = cross_validate(
        pipe, X, y,
        cv=cv10,
        scoring=['accuracy', 'f1_macro', 'f1_weighted'],
        n_jobs=-1
    )

    # Validation LOSO
    scoresloso = cross_validate(
        pipe, X, y,
        cv=loso,
        groups=groups, # <--- TRÈS IMPORTANT si c'est du LeaveOneGroupOut
        scoring=['accuracy', 'f1_macro', 'f1_weighted'],
        n_jobs=-1
    )

    # Stockage des résultats
    results_10fold[name] = {
        'accuracy':    scores10fold['test_accuracy'].mean(),
        'f1_macro':    scores10fold['test_f1_macro'].mean(),
        'f1_weighted': scores10fold['test_f1_weighted'].mean(),
    }

    results_loso[name] = {
        'accuracy':    scoresloso['test_accuracy'].mean(),
        'f1_macro':    scoresloso['test_f1_macro'].mean(),
        'f1_weighted': scoresloso['test_f1_weighted'].mean(),
    }

    r10 = results_10fold[name]
    rl = results_loso[name]

    # Affichage propre en colonnes
    print(f"{name:<18} | {r10['accuracy']:>11.3f} {r10['f1_macro']:>11.3f} {r10['f1_weighted']:>11.3f} | {rl['accuracy']:>10.3f} {rl['f1_macro']:>11.3f} {rl['f1_weighted']:>11.3f}")

# Extraction des meilleurs modèles
best_10fold = max(results_10fold, key=lambda k: results_10fold[k]['f1_macro'])
best_loso = max(results_loso, key=lambda k: results_loso[k]['f1_macro'])

print("\n" + "="*40)
print(f"Meilleur modèle en 10-Fold (Macro F1) : {best_10fold}")
print(f"Meilleur modèle en LOSO (Macro F1)    : {best_loso}")
print("="*40)

## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ?
2. Quel modèle obtient le meilleur F1-score en LOSO ?
3. Les performances chutent-elles en LOSO ? Pourquoi ?
4. Les classes sont-elles équilibrées ?
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ?
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?
7. Quelles améliorations proposeriez-vous ?

1. Le modèle possédant le meilleur F1-score en 10-fold est
2. Le modèle possédant le meilleur F1-score en loso est
3. Oui. cela s'explique par la **variabilité inter-sujet**. En effet, les signaux EEG varient selon le sujet, la position des elctrodes, l'impédance, la fatigue, l'attention, la session d'acquisition et le matériel utilisé (source : cours p102). En 10-fold, le modèle a vu quelques fenêtres de chaque sujet, ce qui lui permet d'adapter sa décision. En LOSO, il n'a aucune information sur le sujet en question.
4. Les classes sont équilibrée, car il aurait fallu appliquer un Stratified K-Fold et que cela ignorerait les déséquilibres des classes.
5. Non, une application multimodale est nécessaire pour compenser le manque de résolution spatiale de l'EEG et filtrer les artefacts musculaires ou environnementaux. En y ajoutant l'ECG, par exemple, on obtient une mesure bien plus fiable et précise de la charge cognitive.
6. L'utilisation du PAAS présente plusieurs limite :
- Deux conducteurs peuvent ressentir différemment la même situation, donc c'est une méthode subjective.
- En le collectant toutes les 10 s, on ignore les variations rapides.
- Répondre pendant la conduite peut modifier la charge cognitive de l'individu.
- Le PAAS est collecté après l'évènement et s'appuie sur la capacité à mémoriser du sujet.
7. Augmenter le nombre de sujets et diversifier les profils (âge, expérience de conduite) pour mieux généraliser.

## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question 

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse 

Si une seule prédiction erronée suffit à déclencher une fausse alerte alors on ne pourrait plus avoir confiance en le système. En effet, puisque les modèles EEG présentent des taux d'erreurs non négligeables, alors le système s'enclencherait trop régulièrement.

In [ ]:
# TODO : utiliser l'exemple dans un dico. Decision system retourne la value de oa key de la charge
# Implique une vérification de la répétition de charge elevée. (Jpense 10*3s). Decision system prend un arg mais j'aimais pas le soulignage rouge donc jai enlevé les ...

# Table de décision
DECISION_TABLE = {
    'charge_faible':             'Interface normale — aucune action requise.',
    'charge_elevee':             'Simplification de l\'interface — réduction des éléments visuels.',
    'charge_elevee_persistante': 'ALERTE CONDUCTEUR — suggestion de pause ou d\'assistance.',
}


def decision_system(predictions, threshold=3):
    """
    Transforme les prédictions en décision d'adaptation.
    
    """
    if not predictions:
        return DECISION_TABLE['charge_faible']

    consecutive = 0
    for pred in reversed(predictions):
        if pred == 1:
            consecutive += 1
        else:
            break

    if consecutive == 0:
        state = 'charge_faible'
    elif consecutive >= threshold:
        state = 'charge_elevee_persistante'
    else:
        state = 'charge_elevee'

    return DECISION_TABLE[state]

print(f'(Seuil persistance : 3 fenêtres consécutives = 30 s)\n')
for label, preds in scenarios:
    decision = decision_system(preds, threshold=3)
    print(f'  {label:<35} → {decision}')



## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features_10s → Normalized_Features_10s/ECG → Normalized_Features_10s_With_Label/ECG
EDA_Features_10s → Normalized_Features_10s/EDA → Normalized_Features_10s_With_Label/EDA
Gaze_Features_10s → Normalized_Features_10s/Gaze → Normalized_Features_10s_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse 

La charge cognitive est multidimensionnelle et se manifeste simultanément dans plusieurs systèmes physiologiques :

- L' **EEG** montre directement l'activité neuronale (bandes theta/alpha/beta), mais est sensible aux artéfacts de mouvement.
- L'**ECG** traduit l'activation du système nerveux autonome sympathique (fréquence cardiaque) lorsque la charge est élevée.
- L'**EDA**  mesure l'activité des glandes sudoripares, indicateur de l'éveil émotionnel et cognitif.
- **Gaze** c'est à dire la dilatation pupillaire et la réduction du taux de clignement sont des permettent de désigner la raison de la charge cognitive.
